# How LLMs see clinical text

*Notebook 1 of 2 — SOBP Session 1*

The point of this notebook is not to teach you machine learning. It's to give you a sturdy enough mental model that, when a vendor shows you their psychiatric LLM product, you can reason about *what it can and can't do, and where it will break*.

Three concepts do most of that work:

1. **Tokenization** — how the model represents the text you give it
2. **Positional encoding** — how it knows what order things came in
3. **Context** — what it actually "knows" at the moment it answers you

We'll touch each one with a clinical example, then end with a demo of why these things matter when you ask an LLM to read a real chart.

In [ ]:
# @title Setup — install deps, authenticate, define `call_llm()` { display-mode: "form" }
# Skip ▶ if you just want to run. The model behind every LLM call below is
# Gemini 2.5 Flash, but the call is wrapped in `call_llm()` so you can swap
# providers in one line.

# ── Install (no-op if already present) ──────────────────────────────────────
%pip install -q -U google-genai tiktoken

import os
from google import genai
from google.genai import types
import tiktoken

# ── Authenticate to Vertex AI ───────────────────────────────────────────────
# In Colab: uses the signed-in Google account.
# Locally:  uses Application Default Credentials (`gcloud auth application-default login`).
try:
    from google.colab import auth as _colab_auth, userdata as _colab_userdata
    _IN_COLAB = True
    _colab_auth.authenticate_user()
except ImportError:
    _IN_COLAB = False

# ── Resolve PROJECT_ID ──────────────────────────────────────────────────────
# Preferred: set `GCP_PROJECT_ID` in Colab Secrets (🔑 left sidebar) or as an
# environment variable. Hardcode below only as a last-resort override.
PROJECT_ID = None
if _IN_COLAB:
    try:
        PROJECT_ID = _colab_userdata.get("GCP_PROJECT_ID")
    except Exception:
        PROJECT_ID = None
PROJECT_ID = PROJECT_ID or os.environ.get("GCP_PROJECT_ID")

if not PROJECT_ID:
    raise RuntimeError(
        "GCP_PROJECT_ID is not set. "
        "In Colab: open the 🔑 Secrets panel (left sidebar) and add a secret "
        "named `GCP_PROJECT_ID` with your project ID, then re-run this cell. "
        "Locally: `export GCP_PROJECT_ID=your-project-id` before launching Jupyter."
    )

LOCATION = "us-central1"
MODEL = "gemini-2.5-flash"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)


def call_llm(prompt: str, system: str | None = None) -> str:
    """Provider-agnostic wrapper. Default backend: Gemini 2.5 Flash via Vertex AI.

    Swap the body of this function to use Anthropic, OpenAI, or a local model
    without touching anything else in the notebook.
    """
    config = types.GenerateContentConfig(system_instruction=system) if system else None
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=config,
    )
    return response.text


# Sanity check.
print(f"Project: {PROJECT_ID}  |  Location: {LOCATION}  |  Model: {MODEL}")


## §1. Tokenization: the model doesn't see words

An LLM doesn't operate on words. It operates on **tokens** — integer IDs from a fixed vocabulary, where each ID corresponds to a sub-word fragment that the model's tokenizer learned during training.

Two implications worth holding onto:

- **Clinical shorthand fragments unpredictably.** Drug names, dosages, ICD codes, abbreviations, and typos all break in different ways.
- **Numerical reasoning starts at a disadvantage.** "100mg" and "100 mg" are different token sequences. The model has to learn that they mean the same thing.

In [ ]:
# We'll use OpenAI's tiktoken to peek at GPT-style tokenization, since it's the
# easiest tokenizer to inspect cell-by-cell. Gemini's tokenizer is similar in
# spirit (BPE-family); the *specific* token boundaries differ, but the lessons
# transfer.

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4-era tokenizer

clinical_sentence = (
    "Pt c/o ↑ anhedonia x 3wks, PHQ-9 = 19, h/o MDD (F33.1), "
    "currently on sertraline 100mg qAM + bupropion XL 300mg, "
    "denies SI/HI, sx refractory despite 8wk adequate trial."
)

token_ids = enc.encode(clinical_sentence)
tokens = [enc.decode([tid]) for tid in token_ids]

print(f"Sentence: {clinical_sentence}\n")
print(f"Word count:  {len(clinical_sentence.split())}")
print(f"Token count: {len(token_ids)}")
print(f"\nTokens (with sub-word boundaries shown as |):")
print("|".join(repr(t) for t in tokens))


Take a look at what the tokenizer did:

- **"sertraline"** got split — likely into something like `sert` + `raline`. The model never sees the word "sertraline" intact; it sees two arbitrary fragments and has to have learned during pretraining that this combination refers to an SSRI.
- **"PHQ-9"** became three or four tokens (`PH`, `Q`, `-`, `9`). The "9" is a generic digit, not bound to the assessment.
- **"F33.1"** fragmented similarly. The ICD-10 structure is invisible to the tokenizer.
- **"100mg"** and **"100 mg"** would tokenize differently. Try it: change the spacing in the sentence and re-run.

None of this is fatal. Modern models handle this routinely. But it explains a lot of small failures — miscounted dosages, mangled abbreviations, fragile performance on rare drug names — and it's the reason "garbage in, garbage out" is sharper for clinical text than for general prose.

*A note for the research-curious.*

Tokenization is a known weak spot, and there's active work to eliminate it. Two recent lines:

- Meta's **Byte Latent Transformer** ([Pagnoni et al., 2024](https://arxiv.org/abs/2412.09871)) operates on raw bytes and learns to group them into variable-sized "patches" based on entropy of the next byte.
- **H-Net** ([Hwang et al., 2025](https://arxiv.org/abs/2507.07955)) learns hierarchical chunking end-to-end from bytes; outperforms a strong BPE Transformer at matched compute, with the largest gains on Chinese, code, and DNA.

Both eliminate the fixed vocabulary entirely. Notably, H-Net uses **state-space models** in its byte-level interface — an alternative to transformers that we won't get into today, but worth knowing the name (you'll hear "Mamba" and "SSM").

None of the major commercial models you can use today (GPT, Claude, Gemini) take this approach yet. They're all BPE-tokenized. But the failure modes we're showing in this notebook are precisely what these lines of work are trying to remove.

## §2. Tokens are a universal adapter

Clinical content arrives in wildly heterogeneous formats: free-text notes, structured medication lists, codes (ICD, RxNorm, LOINC), tables, lab panels, even pasted emails. To work with any of this, you'd normally need format-specific parsers.

**Tokenization sidesteps that problem.** Whatever format you hand the model, it becomes the same kind of object — a sequence of token IDs with positions. The downstream machinery (attention, the next talk) operates on that unified representation indifferently.

Same clinical fact, four ways:

In [ ]:
# Same clinical fact, expressed four ways. We'll tokenize all of them and ask
# the model the same question of each.
#
# Fact: A patient with recurrent MDD started sertraline 50 mg on 2024-03-15,
# titrated to 100 mg on 2024-04-10, with partial response (PHQ-9 18 → 12).

format_A_prose = """Ms. A is a 52-year-old woman with recurrent MDD who presented
on 3/15/24 with PHQ-9 of 18 and was started on sertraline 50 mg daily. At
follow-up on 4/10/24 she was titrated to 100 mg daily. By 5/1/24 her PHQ-9
had improved to 12, consistent with partial response."""

format_B_medlist = """MEDICATIONS — DEPRESSION
  Sertraline  50 mg PO daily   Start: 2024-03-15   Stop: 2024-04-10
  Sertraline 100 mg PO daily   Start: 2024-04-10   Stop: (active)

ASSESSMENTS
  PHQ-9   2024-03-15   Score 18
  PHQ-9   2024-05-01   Score 12"""

format_C_json = """{
  "diagnoses": [{"icd10": "F33.1"}],
  "medications": [
    {"rxnorm": "312940", "name": "sertraline", "dose_mg": 50,
     "start": "2024-03-15", "end": "2024-04-10"},
    {"rxnorm": "314277", "name": "sertraline", "dose_mg": 100,
     "start": "2024-04-10", "end": null}
  ],
  "assessments": [
    {"instrument": "PHQ-9", "date": "2024-03-15", "score": 18},
    {"instrument": "PHQ-9", "date": "2024-05-01", "score": 12}
  ]
}"""

format_D_table = """| Date       | Medication | Dose   | PHQ-9 |
|------------|------------|--------|-------|
| 2024-03-15 | Sertraline | 50 mg  | 18    |
| 2024-04-10 | Sertraline | 100 mg | —     |
| 2024-05-01 | Sertraline | 100 mg | 12    |"""

formats = {
    "A. Narrative prose":     format_A_prose,
    "B. Structured med list": format_B_medlist,
    "C. JSON / coded":        format_C_json,
    "D. Markdown table":      format_D_table,
}

print(f"{'Format':<26} {'Words':>6} {'Tokens':>7}")
print("─" * 42)
for name, txt in formats.items():
    n_words = len(txt.split())
    n_tokens = len(enc.encode(txt))
    print(f"{name:<26} {n_words:>6} {n_tokens:>7}")


In [ ]:
# Now: ask the model the same question of each format. Watch it answer
# correctly from any of them.

question = "What was this patient's PHQ-9 trajectory and what medication change accompanied it?"

for name, txt in formats.items():
    prompt = f"{txt}\n\n---\n\nQuestion: {question}\nAnswer in one sentence."
    answer = call_llm(prompt)
    print(f"\n=== {name} ===\n{answer}\n")


Four totally different surface representations. Four different token sequences. The model answers the same question correctly from any of them.

This is why you don't need a separate parser for "narrative notes vs. structured records vs. coded data vs. tables." Tokenization is the unifying move — once it's done, downstream machinery is format-agnostic.

*(For the neuroscience-inclined audience: this is structurally the same move that libraries like `temporaldata` make for heterogeneous neural recordings — different sessions, rigs, sampling rates all get unified into a single time-indexed representation that downstream models operate on indifferently.)*

## §3. Positional encoding: how the model knows what came in what order

Tokens alone are an unordered bag.

If all the model saw were token IDs, "sertraline 100 mg" and "100 mg sertraline" would be identical input. Word order is *not* in the tokenization.

Order is added separately, as **positional encoding**: each token's vector representation gets a position-specific signal added to it before the model does anything else. Today this is almost always *RoPE* (rotary positional embeddings); historically it was sinusoidal. The math doesn't matter for our purposes.

In [ ]:
# Tokens are a bag without position. Watch:

a = enc.encode("sertraline 100 mg")
b = enc.encode("100 mg sertraline")

print(f"sertraline 100 mg → tokens: {a}")
print(f"100 mg sertraline → tokens: {b}")
print(f"\nSame tokens, different order? {sorted(a) == sorted(b)}")
print(f"Same sequence?                {a == b}")


The useful intuition: **position is information that's added to each token, not something the model gets for free**.

Three consequences worth knowing:

1. **Context windows have a hard ceiling.** A model trained on positions up to *N* doesn't smoothly extrapolate beyond *N*. It degrades, sometimes catastrophically.

2. **Order in the prompt matters.** Instructions placed at the *end* of a long prompt often outweigh instructions placed at the *start*. So do format-defining examples.

3. **The model doesn't attend uniformly across position.** It tends to over-weight the start and end of the input and under-weight the middle. The literature calls this "lost in the middle" ([Liu et al. 2023](https://arxiv.org/abs/2307.03172)).

The next section is a clinical demonstration of #3.

## §4. Context: what the model actually "knows" right now

Everything an LLM "knows" in a given call is one of two things:

1. **Pretraining weights.** Frozen at training time. Doesn't include this morning's news, your patient's chart, or anything that happened to the world after the model was trained.
2. **The context window.** The tokens currently in the prompt. This is the model's entire "working memory" for this call. Nothing else is accessible.

No memory between calls. No filesystem. No database lookup unless you build one (we'll do this in Notebook 2). The window is everything.

The window is finite, and — as we just said — the model doesn't attend uniformly across it. Let's see what that means for clinical work.

In [ ]:
# We'll use a synthetic 28-document psychiatric chart for a fictional patient,
# Eleanor Whitfield, ~12,500 words. The chart is realistic in structure and
# contains a clinically critical detail: a 2022 hospitalization complicated by
# sertraline-induced SIADH with a witnessed seizure, leading to documented
# class-level avoidance of SSRIs.
#
# The CURRENT admission's H&P (Document 24) ends with a Plan that proposes
# initiating sertraline 25 mg daily. Our question: will the model catch this?

import urllib.request

# TODO: replace with raw GitHub URL once the repo is pushed.
CHART_URL = "https://[hosted-location]/whitfield_chart_FULL.md"
chart = urllib.request.urlopen(CHART_URL).read().decode("utf-8")

# Quick stats.
print(f"Words:     {len(chart.split()):,}")
print(f"Tokens:    {len(enc.encode(chart)):,}")
print(f"Documents: {chart.count('## Document')}")


In [ ]:
# The clinical question. This is the prompt we'd actually want a clinical-AI
# assistant to answer well.

question = """You are reviewing this patient's chart prior to attending rounds.
Are there any safety concerns with the current medication plan documented
in the most recent admission H&P? If so, describe them specifically and
explain what you would recommend."""

prompt = f"{chart}\n\n---\n\n{question}"
answer = call_llm(prompt)
print(answer)


Whether the model caught it depends on (a) how the chart was packed into the context window and (b) how attention is distributed across position.

Here's the more revealing experiment:

In [ ]:
# Same chart, same question, but we move the 2022 discharge summary
# (Document 12 — the most detailed account of the SSRI reaction) to three
# different positions: at the start, in the middle (where it natively lives),
# and at the end.

import re

def split_documents(chart_text: str) -> dict[int, str]:
    """Split the chart on `## Document N — ...` headers.

    Returns {doc_num: full_doc_text_including_header}.
    """
    # Capture each document block from its header up to the next header (or EOF).
    pattern = r"(## Document (\d+) —[^\n]*\n.*?)(?=\n## Document \d+ —|\Z)"
    matches = re.findall(pattern, chart_text, flags=re.DOTALL)
    return {int(num): block.strip() for block, num in matches}


docs = split_documents(chart)
dc_summary = docs[12]                              # 2022 Discharge Summary
chart_minus_dc = chart.replace(dc_summary, "").strip()

prompts = {
    "needle at START":         f"{dc_summary}\n\n{chart_minus_dc}",
    "needle in MIDDLE (native)": chart,            # already in middle
    "needle at END":           f"{chart_minus_dc}\n\n{dc_summary}",
}

for label, prompt_text in prompts.items():
    full_prompt = f"{prompt_text}\n\n---\n\n{question}"
    answer = call_llm(full_prompt)
    print(f"\n{'=' * 60}\n{label}\n{'=' * 60}\n{answer[:600]}...\n")


A few takeaways from what just happened:

1. **The chart contained the answer.** In every position, the same words were in the same context window. The model "had access to" the SSRI contraindication in all three runs.

2. **Access is not retrieval.** What the model surfaces depends on where in the window the relevant tokens sit, what's near them, and how attention happens to be distributed for that prompt and that model.

3. **For clinical use this is the central limitation.** A long chart is the *normal* case, not the edge case. A real EHR pull for a complex patient runs 50,000–100,000+ words. Asking an LLM to "review the chart" without engineering around the context window is fragile.

4. **This is what motivates RAG** — retrieval-augmented generation, in Notebook 2. Instead of cramming the whole chart in and hoping, RAG retrieves the *relevant* chunks first and only puts those in front of the model.

## What's next

We've covered three things:

- **Tokenization** is how text becomes the model's input — fragments of words as integers, with all the messiness that creates.
- **Positional encoding** is how the model knows what came in what order, and why "lost in the middle" happens.
- **Context** is the model's entire working memory for a given call — finite, non-uniform, and the source of most clinical failures.

What we *haven't* covered is what the model actually does with the tokens once it has them. That's **attention** — the operation that lets each token in the window influence each other token. Attention is the heart of the transformer, and it's the subject of the next talk in this session.

---

*Notebook 2 picks up the practical thread: how to use these systems for psychiatric research, given what we now know about how they work — and where they break.*